### This is for extraction of data from catalogues like FIRST, NVSS, LoTSS, GLEAM, etc. 

### The data may be of FITS, or CSV or VoTable format or any other as per requirements.


### Aim: To prepare a extraction and preprocessing pipeline for developement of ML models for:

### 1. Morphological classification
### 2. Source Detection and classification
### 3. Transient detection, discovery and classification
### 4. Spectral classification, analysis and detection.

In [ ]:
# import io
# import os
# import numpy as np
# import requests
# from astropy import coordinates
# from astropy import units as u
# from astropy.io import fits

# # NASA SkyView's FIRST 1.4 GHz cutout service (the FIRST cutout server third.ucllnl.org was unreachable)
# SKYVIEW_URL = "https://skyview.gsfc.nasa.gov/cgi-bin/pskcall"
# FIRST_SURVEY = "vla first (1.4 ghz)"
# FIRST_PIXEL_SCALE = 1.8 * u.arcsec  # native FIRST pixel size

# # 1. Define your list of target coordinates
# target_coords = [
#     coordinates.SkyCoord('12h29m06.7s +02d03m08s', frame='icrs'), # 3C 273
#     coordinates.SkyCoord('13h30m37.7s +11d16m29s', frame='icrs'),
#     coordinates.SkyCoord(202.45, 47.12, unit=(u.deg, u.deg), frame='icrs')
# ]

# # 2. Set your image size (3 arcmin -> 100 x 100 pixels at the native FIRST scale)
# cutout_size = 3 * u.arcmin
# npix = int(round((cutout_size / FIRST_PIXEL_SCALE).decompose().value))

# # 3. Create a directory to store the FITS files
# output_dir = r"I:\Radio and Gamma-ray observational astronomy Data analysis projects\R1\FIRST_FITS_CUTOUTS"
# os.makedirs(output_dir, exist_ok=True)

# # 4. Loop through coordinates and download (fail fast: 10 s to connect, 60 s to read, 2 attempts)
# for i, coord in enumerate(target_coords):
#     print(f"Fetching cutout for Target {i+1} at RA={coord.ra.deg:.4f}, Dec={coord.dec.deg:.4f}...")
    
#     params = {
#         "position": f"{coord.ra.deg},{coord.dec.deg}",
#         "survey": FIRST_SURVEY,
#         "pixels": f"{npix},{npix}",
#         "size": cutout_size.to(u.deg).value,
#         "projection": "Tan",
#         "coordinates": "J2000",
#         "return": "FITS",
#     }
    
#     for attempt in range(1, 3):
#         try:
#             response = requests.get(SKYVIEW_URL, params=params, timeout=(10, 60))
#             response.raise_for_status()
            
#             # A bad request returns an HTML error page, which fits.open rejects
#             with fits.open(io.BytesIO(response.content)) as hdulist:
#                 # Outside the footprint (or with a wrong survey name) SkyView still returns a FITS file:
#                 # all zeros, or without SURVEY=FIRST
#                 in_footprint = hdulist[0].header.get("SURVEY") == "FIRST" and np.any(np.nan_to_num(hdulist[0].data))
            
#             if not in_footprint:
#                 print(f"  -> No data found in FIRST footprint for these coordinates.")
#                 break
            
#             # Save the bytes exactly as served (the FIRST header has tab characters that astropy's strict writer rejects)
#             filename = f"FIRST_J{coord.ra.deg:.4f}_{coord.dec.deg:.4f}.fits"
#             filepath = os.path.join(output_dir, filename)
#             with open(filepath, "wb") as f:
#                 f.write(response.content)
#             print(f"  -> Successfully saved to {filepath}")
#             break
            
#         except Exception as e:
#             print(f"  -> Attempt {attempt} failed: {e}")
#     else:
#         print(f"  -> Giving up on Target {i+1}.")

# print("Download complete!")

Above test completed. That test is for extraction of fits file from FIRST.

In [1]:
import io
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pyvo
import requests
from astropy import units as u
from astropy.io import fits

# NASA SkyView's FIRST 1.4 GHz cutout service
SKYVIEW_URL = "https://skyview.gsfc.nasa.gov/cgi-bin/pskcall"
FIRST_SURVEY = "vla first (1.4 ghz)"
FIRST_PIXEL_SCALE = 1.8 * u.arcsec  # native FIRST pixel size

# 1. Settings (4.5 arcmin -> 100 x 100 pixels at the native FIRST scale)
n_sources = 10000
n_workers = 8  # SkyView answers "Too Many Processes" to about half the requests at 50 at once
cutout_size = 4.5 * u.arcmin
npix = int(round((cutout_size / FIRST_PIXEL_SCALE).decompose().value))
output_dir = r"I:\Radio and Gamma-ray observational astronomy Data analysis projects\R1\FIRST_FITS_CUTOUTS\FIRST"
os.makedirs(output_dir, exist_ok=True)

# 2. Query the FIRST catalogue (VizieR VIII/92/first14) through TAP.
# The catalogue is sorted by declination (its first rows are all at Dec +64), so take every step-th row
# to get n_sources sources spread over the whole survey, and the same sources on every run
tap = pyvo.dal.TAPService("http://tapvizier.cds.unistra.fr/TAPVizieR/tap")
n_total = int(tap.run_sync('SELECT COUNT(*) AS n FROM "VIII/92/first14"').to_table()["n"][0])
step = n_total // n_sources
catalog = tap.run_sync(
    f'SELECT TOP {n_sources} recno, FIRST, RAJ2000, DEJ2000, Fpeak, Fint, Rms, Maj, Min, PA '
    f'FROM "VIII/92/first14" WHERE MOD(recno, {step}) = 0 ORDER BY recno'
).to_table()
catalog.write(os.path.join(output_dir, "FIRST_10000_catalog.csv"), format="csv", overwrite=True)
print(f"Selected {len(catalog)} of {n_total} FIRST sources (every {step}th row)")

# 3. Download one cutout (10 s to connect, 60 s to read, 3 attempts with growing waits:
# SkyView answers a busy period with an HTML page, status 200, instead of a FITS file)
def fetch_cutout(source):
    name, ra, dec = source
    filepath = os.path.join(output_dir, f"FIRST_{name}.fits")
    if os.path.exists(filepath):  # delete a file to fetch it again
        return name, "exists", ""
    
    params = {
        "position": f"{ra},{dec}",
        "survey": FIRST_SURVEY,
        "pixels": f"{npix},{npix}",
        "size": cutout_size.to(u.deg).value,
        "projection": "Tan",
        "coordinates": "J2000",
        "return": "FITS",
    }
    error = ""
    for attempt in range(3):
        if attempt:
            time.sleep(5 * attempt)  # wait 5 s, then 10 s
        try:
            response = requests.get(SKYVIEW_URL, params=params, timeout=(10, 60))
            response.raise_for_status()
            
            # Show the server's message (HTML tags removed) when it did not send a FITS file
            if not response.content.startswith(b"SIMPLE"):
                message = " ".join(re.sub(r"<[^>]+>", " ", response.text).split())
                raise ValueError(f"not FITS, server sent: {message[:120]}")
            with fits.open(io.BytesIO(response.content)) as hdulist:
                # Outside the footprint SkyView still returns a FITS file: all zeros
                in_footprint = hdulist[0].header.get("SURVEY") == "FIRST" and np.any(np.nan_to_num(hdulist[0].data))
            if not in_footprint:
                return name, "no data", ""
            
            # Save the bytes exactly as served (the FIRST header has tab characters that astropy's strict writer rejects)
            with open(filepath, "wb") as f:
                f.write(response.content)
            return name, "saved", ""
        except Exception as e:
            error = str(e)[:150]
    return name, "failed", error

# 4. Download all cutouts, n_workers at a time
counts = {}
sources = zip(catalog["FIRST"], catalog["RAJ2000"], catalog["DEJ2000"])
with ThreadPoolExecutor(max_workers=n_workers) as pool:
    for done, (name, status, error) in enumerate(pool.map(fetch_cutout, sources), start=1):
        counts[status] = counts.get(status, 0) + 1
        if status in ("no data", "failed"):
            print(f"  -> {name}: {status} {error}")
        if done % 250 == 0:
            print(f"  {done}/{len(catalog)} processed {counts}")

print(f"Download complete! {counts}")
if counts.get("failed"):
    print("Run this cell again to retry the failed ones (existing files are skipped).")

Selected 10000 of 946432 FIRST sources (every 94th row)
  250/10000 processed {'saved': 250}
  500/10000 processed {'saved': 500}
  750/10000 processed {'saved': 750}
  1000/10000 processed {'saved': 1000}
  1250/10000 processed {'saved': 1250}
  1500/10000 processed {'saved': 1500}
  1750/10000 processed {'saved': 1750}
  2000/10000 processed {'saved': 2000}
  2250/10000 processed {'saved': 2250}
  2500/10000 processed {'saved': 2500}
  2750/10000 processed {'saved': 2750}
  3000/10000 processed {'saved': 3000}
  3250/10000 processed {'saved': 3250}
  3500/10000 processed {'saved': 3500}
  3750/10000 processed {'saved': 3750}
  4000/10000 processed {'saved': 4000}
  4250/10000 processed {'saved': 4250}
  4500/10000 processed {'saved': 4500}
  4750/10000 processed {'saved': 4750}
  5000/10000 processed {'saved': 5000}
  5250/10000 processed {'saved': 5250}
  5500/10000 processed {'saved': 5500}
  5750/10000 processed {'saved': 5750}
  6000/10000 processed {'saved': 6000}
  6250/10000 p